# ARIA Role and Attribute Explainer

The developer pastes a snippet of HTML using ARIA roles, states, and properties and asks whether it is correct. The agent reviews the markup, identifies any invalid role and attribute combinations, missing required attributes, or redundant ARIA that duplicates native semantics, and explains each issue in plain language with a corrected version. It then asks follow-up questions about the component's dynamic behavior — does the expanded state change, does the selected state update, does focus move on activation — and generates the JavaScript needed to keep ARIA states in sync with the UI. The conversation continues until the developer has a complete, correct, and accessible implementation.

In [ ]:
# imports

import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [ ]:
load_dotenv(override=True)
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')

if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set")

anthropic_url = "https://api.anthropic.com/v1/"
anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)
MODEL = "claude-sonnet-4-6"

## Step 1: Define the system prompt

The system prompt establishes the agent's expert persona, a structured response format for each turn, and explicit memory behavior across turns.

Key requirements:
- **Expert persona**: senior accessibility engineer fluent in the ARIA spec, the ARIA Authoring Practices Guide (APG), and how screen readers interpret roles and attributes
- **Structured response format** for each turn:
  1. ARIA audit — categorized issues (invalid combination, missing required attribute, redundant ARIA, broken relationship)
  2. Plain language explanation of each issue and its screen reader impact
  3. Corrected markup in a fenced code block
  4. Exactly one follow-up question about dynamic behavior
- **JavaScript generation** triggered after the developer answers the follow-up: which attributes to toggle, which events trigger the toggle, and a concrete event listener example
- **Memory behavior**: track the component type across turns, surface systematic patterns when the same error recurs, and adapt code examples to the developer's visible framework (React, Vue, or vanilla HTML/JS)

In [ ]:
system_prompt = """You are a senior web accessibility engineer specializing in ARIA (Accessible Rich Internet Applications). Your job is to review HTML markup that uses ARIA roles, states, and properties, identify problems, and help developers reach a complete, correct, and accessible implementation through multi-turn dialogue.

When a developer pastes an HTML snippet, structure your response as follows:

**ARIA Audit** — List every issue found in the markup. For each issue, identify the category:
- ❌ Invalid role/attribute combination — an attribute that is not in the supported state and property list for the role (e.g., `aria-checked` on `role="listbox"`)
- ⚠️ Missing required attribute — an attribute the ARIA spec requires for this role to function correctly (e.g., `role="combobox"` without `aria-expanded` or `aria-controls`)
- 🔁 Redundant ARIA — markup that duplicates native HTML semantics (e.g., `role="button"` on a native `<button>` element, or `aria-label` on an `<img>` that already has equivalent `alt` text)
- 🔗 Broken ARIA relationship — an `aria-labelledby`, `aria-describedby`, `aria-controls`, or `aria-owns` value pointing to an ID that does not exist in the snippet

If no issues are found, say so explicitly and explain why the markup is correct.

**Plain language explanation** — For each issue listed, explain in one or two sentences what the problem is and how it affects assistive technology users — specifically what a screen reader would announce incorrectly or fail to announce.

**Corrected markup** — Provide the corrected HTML in a fenced code block. Change only what is necessary to fix the ARIA issues. Do not reformat, restructure, or restyle the markup.

**Dynamic behavior follow-up** — Ask exactly one question about the component's runtime behavior that determines what JavaScript state management is needed. Choose the most important unknown from this list:
- Does the expanded/collapsed state change? (accordions, disclosures, dropdowns)
- Does the selected/checked state update? (tabs, radio groups, checkboxes)
- Does focus move on activation? (modals, dialogs, drawers)
- Is this a live region, and if so, when is content added to it?
- Does the component manage its own focus, or does the page handle it?

**JavaScript generation (after follow-up)** — Once the developer answers the follow-up question, generate the JavaScript needed to keep ARIA states in sync with the UI:
1. Which attributes to toggle and which events trigger the toggle
2. A minimal, self-contained event listener implementation with inline comments explaining each step
3. Any cleanup needed (removing listeners, resetting state on close)

Provide the JavaScript in a fenced code block with the correct language tag — javascript for plain HTML, jsx for React, or the appropriate framework syntax based on what the developer's code uses.

Memory behavior across turns:
- Track the component type established in the first turn. If the developer pastes a follow-up snippet for the same component, reference the earlier audit by name.
- If the same incorrect pattern appears more than once in the session, name the systematic issue explicitly and suggest a shared fix at the design system or component library level.
- Adapt code examples to the framework visible in the developer's snippets — React JSX and useState hooks, Vue 3 composition API, or vanilla JavaScript setAttribute.
- After generating the JavaScript, always ask: "Does this cover all the states and interactions for this component, or are there additional behaviors to address?" Do not treat the session as complete until the developer confirms."""

## Step 2: Write a Gradio chat callback (placeholder)

Gradio's `ChatInterface` expects a callback with the signature:

```python
def chat(message, history):
    ...
```

- `message` — the string the user just typed
- `history` — a list of prior turns, each a dict with `"role"` (`"user"` or `"assistant"`) and `"content"` keys

The callback must **return** (or **yield**, for streaming) the assistant's reply as a plain string.

Start with a hardcoded placeholder return value to confirm the Gradio UI launches and the callback wiring works before connecting the LLM.

In [ ]:
def chat(message, history):
    return "I can see your HTML. Let me audit it for ARIA issues."

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

## Step 3: Implement the real chat callback

Replace the placeholder with a real implementation that calls the Anthropic API via the OpenAI-compatible client.

The key pattern:

1. **Convert history**: Gradio gives you `history` as a list of `{"role": ..., "content": ...}` dicts — select only the keys the API needs
2. **Build the messages list**: prepend the system message and append the current user message
   ```python
   messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
   ```
3. **Call the API** using the `anthropic` client (OpenAI SDK pointed at Anthropic's endpoint) and return `response.choices[0].message.content`

```python
def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = anthropic.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content
```

Note: Gradio passes `history` *without* the current message — that's why we append the current `message` ourselves at the end.

In [ ]:
def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = anthropic.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

## Step 4: Add streaming

Instead of waiting for the full response before displaying anything, **streaming** yields partial tokens as they arrive. This matters for ARIA audits because the agent's responses can be long — an audit plus corrected markup plus a follow-up question may be several hundred tokens.

Change `return` to `yield` and pass `stream=True` to the same `anthropic` client:

```python
def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    stream = anthropic.chat.completions.create(model=MODEL, messages=messages, stream=True)
    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ""
        yield response
```

Gradio's `ChatInterface` handles generators automatically — it updates the chat bubble incrementally as each `yield` fires.

To switch to a different provider later, change `anthropic` to another OpenAI-compatible client and update `MODEL`.

In [ ]:
def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]

    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]

    stream = anthropic.chat.completions.create(model=MODEL, messages=messages, stream=True)

    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ""
        yield response

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

## Step 5: Dynamic system message based on detected component type

The pattern for a dynamic system message is:

```python
relevant_system_prompt = system_prompt
if "belt" in message.lower():
    relevant_system_prompt += " The store does not sell belts..."
messages = [{"role": "system", "content": relevant_system_prompt}] + history + [{"role": "user", "content": message}]
```

Apply the same pattern to the ARIA domain: when the developer's HTML contains a specific ARIA role, inject targeted, authoritative rules for that role into the system message so the agent's advice is more precise.

Examples of role-specific additions:
- `role="dialog"` or `role="alertdialog"` → append rules about required `aria-modal`, `aria-labelledby`, and focus management on open/close
- `role="combobox"` → append the required attribute list (`aria-expanded`, `aria-controls`, `aria-activedescendant`) and the owned-element pattern
- `role="tab"` → append rules about `aria-selected`, `aria-controls` on tabs, and `role="tabpanel"` with `aria-labelledby` on panels

Detect roles by checking whether the role string appears in `message` (which contains the pasted HTML). Build the relevant additions and prepend `relevant_system_prompt` as the system message.

In [ ]:
# TODO: implement chat(message, history) with a dynamic system message
#
# ROLE_ADDITIONS is a dict mapping a detectable string to extra instructions.
# Add entries for the ARIA roles you want to handle.
#
# ROLE_ADDITIONS = {
#     'role="dialog"': (
#         " Additional rules for dialogs: the element must have aria-modal=\"true\" "
#         "if it is rendered in a layer above other page content. It must have an "
#         "accessible name via aria-labelledby (preferred) or aria-label. Focus must "
#         "move to the dialog or its first focusable descendant on open, and return "
#         "to the trigger element on close."
#     ),
#     'role="combobox"': (
#         " Additional rules for comboboxes: aria-expanded (true/false) and "
#         "aria-controls (pointing to the owned listbox id) are both required. "
#         "aria-activedescendant must be set to the id of the currently focused "
#         "option when the listbox is open. The input element is the combobox — "
#         "do not put role=\"combobox\" on a wrapper div."
#     ),
#     'role="tab"': (
#         " Additional rules for tabs: each tab must have aria-selected (true/false) "
#         "and aria-controls pointing to its associated tabpanel id. Each tabpanel "
#         "must have aria-labelledby pointing back to its tab id. Only the active "
#         "tab should have aria-selected=\"true\"; all others must be \"false\"."
#     ),
# }
#
# def chat(message, history):
#     # Step 1: start with the base system prompt
#     relevant_system_prompt = system_prompt
#
#     # Step 2: detect ARIA roles in the pasted HTML and append role-specific rules
#     for role_string, addition in ROLE_ADDITIONS.items():
#         if role_string in message:
#             relevant_system_prompt += addition
#
#     # Step 3: convert history
#     history = [{"role": h["role"], "content": h["content"]} for h in history]
#
#     # Step 4: build messages list using the dynamic system prompt
#     messages = [{"role": "system", "content": relevant_system_prompt}] + history + [{"role": "user", "content": message}]
#
#     # Step 5: stream and yield
#     stream = anthropic.chat.completions.create(model=MODEL, messages=messages, stream=True)
#     response = ""
#     for chunk in stream:
#         response += chunk.choices[0].delta.content or ""
#         yield response

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

## Sample HTML snippets to test with

Use these to exercise the agent across the four ARIA issue categories. Paste them into the Gradio chat window.

**Test 1 — Invalid role/attribute combination**
```html
<ul role="listbox" aria-checked="false">
  <li role="option">Option A</li>
  <li role="option">Option B</li>
</ul>
```
`aria-checked` is valid on `role="checkbox"`, `role="menuitemcheckbox"`, and `role="treeitem"` — not on `role="listbox"`. The correct attribute for a listbox option is `aria-selected`.

---

**Test 2 — Missing required attributes**
```html
<div role="combobox">
  <input type="text" />
  <ul role="listbox">
    <li role="option">Apple</li>
    <li role="option">Banana</li>
  </ul>
</div>
```
`role="combobox"` requires `aria-expanded` and `aria-controls`. The listbox also has no accessible name.

---

**Test 3 — Redundant ARIA**
```html
<button role="button" aria-label="Submit form">Submit</button>
```
`role="button"` is redundant on a native `<button>`. The `aria-label` overrides the visible text, creating a WCAG 2.5.3 (Label in Name) failure.

---

**Test 4 — Broken ARIA relationship**
```html
<button aria-controls="menu-dropdown" aria-expanded="false">Menu</button>
<ul id="nav-menu" role="menu">
  <li role="menuitem">Home</li>
  <li role="menuitem">About</li>
</ul>
```
`aria-controls="menu-dropdown"` points to an ID that does not exist — the `<ul>` has `id="nav-menu"`.